# E06


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import tempfile
from pathlib import Path


IN_COLAB = Path('/content').exists()
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MYDRIVE.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive was not mounted automatically:', repr(exc))

import cv2


def find_supplemental_root() -> Path:
    explicit = globals().get('SYNISCOPY_SUPPLEMENTAL_ROOT', None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    if DRIVE_MYDRIVE.exists():
        candidates.extend([
            DRIVE_MYDRIVE / 'supplemental',
            DRIVE_MYDRIVE / 'SyniscopySupplemental',
        ])
    candidates.extend([Path('/content/supplemental'), Path('/content/SyniscopySupplemental')])
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if (resolved / 'supplemental' / 'E01.ipynb').exists():
            resolved = resolved / 'supplemental'
        if (resolved / 'E01.ipynb').exists():
            return resolved
    raise RuntimeError('Syniscopy supplemental folder not found. Upload supplemental/ to Drive as MyDrive/supplemental.')


def resolve_supplemental_path(value: str | os.PathLike[str]) -> Path:
    path = Path(value).expanduser()
    if path.is_absolute():
        return path
    candidates = [
        SUPPLEMENTAL_ROOT / path,
        SUPPLEMENTAL_ROOT.parent / path,
        Path.cwd() / path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return SUPPLEMENTAL_ROOT / path


def supplemental_relative(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(SUPPLEMENTAL_ROOT.resolve()))
    except Exception:
        try:
            return str(path.resolve().relative_to(SUPPLEMENTAL_ROOT.parent.resolve()))
        except Exception:
            return str(path)


def _ffprobe_video_info(video_path: Path) -> tuple[int | None, float | None]:
    try:
        raw = subprocess.check_output(
            [
                'ffprobe', '-v', 'error', '-select_streams', 'v:0',
                '-show_entries', 'stream=nb_frames,r_frame_rate,duration',
                '-of', 'json', str(video_path),
            ],
            text=True,
        )
    except Exception:
        return None, None
    try:
        stream = (json.loads(raw).get('streams') or [{}])[0]
    except Exception:
        return None, None
    frame_count = None
    nb_frames = stream.get('nb_frames')
    if nb_frames not in (None, 'N/A'):
        try:
            frame_count = int(nb_frames)
        except Exception:
            frame_count = None
    fps = None
    rate = stream.get('r_frame_rate')
    if rate and rate != '0/0':
        try:
            num, den = rate.split('/')
            fps = float(num) / float(den)
        except Exception:
            fps = None
    if fps is None and stream.get('duration') and frame_count:
        try:
            fps = float(frame_count) / float(stream['duration'])
        except Exception:
            fps = None
    return frame_count, fps


def video_info(video_path: Path) -> tuple[int | None, float | None]:
    cap = cv2.VideoCapture(str(video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if cap.isOpened() else None
    fps = float(cap.get(cv2.CAP_PROP_FPS)) if cap.isOpened() else None
    cap.release()
    if frame_count is not None and frame_count <= 0:
        frame_count = None
    if fps is not None and fps <= 0:
        fps = None
    probe_count, probe_fps = _ffprobe_video_info(video_path)
    return frame_count if frame_count is not None else probe_count, fps if fps is not None else probe_fps


def read_frame(video_path: Path, frame_idx: int):
    cap = cv2.VideoCapture(str(video_path))
    ok = False
    frame = None
    if cap.isOpened():
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
    cap.release()
    if ok and frame is not None:
        return frame
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_frame = Path(tmpdir) / 'prompt_frame.png'
        subprocess.run(
            [
                'ffmpeg', '-y', '-loglevel', 'error', '-i', str(video_path),
                '-vf', f'select=eq(n\\,{frame_idx})', '-frames:v', '1', str(tmp_frame),
            ],
            check=True,
        )
        frame = cv2.imread(str(tmp_frame), cv2.IMREAD_COLOR)
    if frame is None:
        raise RuntimeError(f'Could not decode prompt frame {frame_idx} from {video_path}')
    return frame


def normalize_points(raw_points) -> list[dict]:
    points = []
    for point in raw_points or []:
        points.append({
            'x': float(point['x']),
            'y': float(point['y']),
            'label': point.get('label', 'particle'),
            'point_label': int(point.get('point_label', 1)),
        })
    return points


def normalize_boxes(raw_boxes) -> list[dict]:
    boxes = []
    for box in raw_boxes or []:
        boxes.append({
            'x1': float(box['x1']),
            'y1': float(box['y1']),
            'x2': float(box['x2']),
            'y2': float(box['y2']),
            'label': box.get('label', 'particle'),
        })
    return boxes


def draw_prompt_preview(video_path: Path, frame_idx: int, points: list[dict], boxes: list[dict], out: Path) -> None:
    frame = read_frame(video_path, frame_idx)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    preview = frame_rgb.copy()
    for point in points:
        x, y = int(round(point['x'])), int(round(point['y']))
        if int(point.get('point_label', 1)) == 1:
            cv2.circle(preview, (x, y), 5, (255, 0, 0), -1)
            cv2.drawMarker(preview, (x, y), (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=14, thickness=1)
        else:
            cv2.circle(preview, (x, y), 4, (0, 96, 255), 1)
            cv2.drawMarker(preview, (x, y), (0, 96, 255), markerType=cv2.MARKER_TILTED_CROSS, markerSize=12, thickness=1)
    for box in boxes:
        x1, y1, x2, y2 = [int(round(float(box[key]))) for key in ('x1', 'y1', 'x2', 'y2')]
        cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 1)
    out.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(out), cv2.cvtColor(preview, cv2.COLOR_RGB2BGR)):
        raise RuntimeError(f'Failed to write prompt preview: {out}')


SUPPLEMENTAL_ROOT = find_supplemental_root()
OUTPUT_DIR = SUPPLEMENTAL_ROOT / 'outputs' / 'E06'
PREVIEW_DIR = OUTPUT_DIR / 'prompt_previews'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
for old_preview in PREVIEW_DIR.glob('*.png'):
    old_preview.unlink()
(OUTPUT_DIR / 'sam2-caustic-prompt.png').unlink(missing_ok=True)

explicit_video = globals().get('SYNISCOPY_E06_VIDEO', None) or os.environ.get('SYNISCOPY_E06_VIDEO')
explicit_manifest = (
    globals().get('SYNISCOPY_E06_SELECTED_CLIP_MANIFEST', None)
    or os.environ.get('SYNISCOPY_E06_SELECTED_CLIP_MANIFEST')
)
default_manifest = SUPPLEMENTAL_ROOT / 'data' / 'liverpool_caustic_50nm_review' / 'selected_clip_manifest.json'

clip_rows = []
source_manifest = None
if explicit_video:
    video_path = resolve_supplemental_path(explicit_video)
    clip_rows = [{
        'clip_video': str(video_path),
        'prompt_frame_idx': 0,
        'points': [{'x': 44.0, 'y': 171.0, 'label': 'particle', 'point_label': 1}],
        'boxes': [{'x1': 16.0, 'y1': 143.0, 'x2': 72.0, 'y2': 195.0, 'label': 'particle'}],
        'source_video': str(video_path),
    }]
elif explicit_manifest or default_manifest.exists():
    source_manifest = resolve_supplemental_path(explicit_manifest) if explicit_manifest else default_manifest
    if not source_manifest.exists():
        raise FileNotFoundError(f'Missing selected clip manifest: {source_manifest}')
    clip_rows = json.loads(source_manifest.read_text(encoding='utf-8'))
else:
    raise FileNotFoundError(
        'Missing reviewed clip manifest. Export reviewed caustic clips to supplemental/data/liverpool_caustic_50nm_review first, '
        'or set SYNISCOPY_E06_VIDEO to a local AVI path.'
    )

if not clip_rows:
    raise RuntimeError('No clips available for E06 prompt manifest generation.')

clips = []
for idx, row in enumerate(clip_rows):
    video_path = resolve_supplemental_path(row['clip_video'])
    if not video_path.exists():
        raise FileNotFoundError(f'Missing E06 clip video: {video_path}')
    prompt_frame_idx = int(row.get('prompt_frame_idx', 0))
    points = normalize_points(row.get('points', []))
    boxes = normalize_boxes(row.get('boxes', []))
    if not points:
        raise RuntimeError(f'Missing prompt points for clip row {idx}: {video_path}')
    frame_count, fps = video_info(video_path)
    preview_path = PREVIEW_DIR / f'{video_path.stem}_prompt.png'
    draw_prompt_preview(video_path, prompt_frame_idx, points, boxes, preview_path)
    clip_entry = {
        'clip_index': idx,
        'source_video': row.get('source_video', row.get('working_video', row.get('clip_video'))),
        'clip_video': supplemental_relative(video_path),
        'frame_count': frame_count,
        'fps': fps,
        'fps_metadata': fps,
        'fps_expected_from_repository': row.get('fps_expected_from_repository'),
        'analysis_fps': row.get('fps_expected_from_repository') or fps,
        'pixel_size_nm': row.get('pixel_size_nm'),
        'particle_diameter_nm': row.get('particle_diameter_nm'),
        'source_frame_start': row.get('source_frame_start'),
        'source_frame_end_exclusive': row.get('source_frame_end_exclusive'),
        'clip_frame_count': row.get('clip_frame_count', frame_count),
        'prompt_preview_png': supplemental_relative(preview_path),
        'audit_json': row.get('audit_json'),
        'prompts': [
            {
                'frame_idx': prompt_frame_idx,
                'frame_path': f'extracted frame {prompt_frame_idx}',
                'points': points,
                'boxes': boxes,
            }
        ],
    }
    clips.append(clip_entry)

manifest = {
    'schema_version': 'syniscopy-e06-sam2-prompt-v3',
    'source_manifest': supplemental_relative(source_manifest) if source_manifest else None,
    'clip_count': len(clips),
    'clips': clips,
    'sam2_training_notebook': 'E08.ipynb',
    'sam2_inference_notebook': 'E09.ipynb',
    'sam2_inference_outputs': 'outputs/E09/inference_outputs/<video-stem>/<variant>/',
    'sam2_inference_variants': ['base', 'finetuned'],
}

# Backward-compatible top-level fields for tools that expect the older one-clip E06 manifest.
first = clips[0]
legacy_preview_path = OUTPUT_DIR / 'sam2-caustic-prompt.png'
shutil.copy2(resolve_supplemental_path(first['prompt_preview_png']), legacy_preview_path)
manifest.update({
    'source_video': first['clip_video'],
    'frame_count': first['frame_count'],
    'fps': first['fps'],
    'fps_metadata': first.get('fps_metadata', first['fps']),
    'fps_expected_from_repository': first.get('fps_expected_from_repository'),
    'analysis_fps': first.get('analysis_fps'),
    'pixel_size_nm': first.get('pixel_size_nm'),
    'particle_diameter_nm': first.get('particle_diameter_nm'),
    'prompt_preview_png': supplemental_relative(legacy_preview_path),
    'prompts': first['prompts'],
})

manifest_path = OUTPUT_DIR / 'sam2-caustic-transfer-manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True, allow_nan=False), encoding='utf-8')
print(manifest_path)
print(f'Wrote {len(clips)} prompt records from {source_manifest or "explicit video"}')
for clip in clips[:5]:
    print(f"  [{clip['clip_index']:02d}] {clip['clip_video']} -> {clip['prompt_preview_png']}")
if len(clips) > 5:
    print(f'  ... {len(clips) - 5} more clips')
